# Feature IC Analysis

Information Coefficient (IC) analysis for all candidate features vs `forward_return_1y`.  
Uses sector-neutral IC with Newey-West t-stats and Benjamini-Hochberg FDR correction.

**Packages used**: `research.ic_engine`, `alpha.horizon_router` — proving Architecture V2 works for real research.

In [ ]:
import sys
from pathlib import Path

# Add project root to path for notebook execution
sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from research.ic_engine import compute_yearly_ic, newey_west_tstat, bh_fdr_correction
from modeling.train import get_candidates, EXCLUDE, EXCLUDE_PATTERNS
from alpha.horizon_router import FEATURE_FACTOR_GROUPS

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", font_scale=0.9)

DATA_PATH = Path("../data/historical_dataset_clean.parquet")
RET_COL = "forward_return_1y"

## 1. Load Data

In [ ]:
if not DATA_PATH.exists():
    print(f"ERROR: Dataset not found at {DATA_PATH.resolve()}")
    print("Download it or run the pipeline first. Exiting gracefully.")
    sys.exit(0)

df = pd.read_parquet(DATA_PATH)
print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Fiscal years: {int(df['fiscal_year'].min())}–{int(df['fiscal_year'].max())}")
print(f"Forward return coverage: {df[RET_COL].notna().mean():.1%}")

## 2. Feature Coverage Table

In [ ]:
candidates = get_candidates(df)
print(f"Candidate features: {len(candidates)}")

coverage = pd.DataFrame({
    "fill_rate": df[candidates].notna().mean(),
    "mean": df[candidates].mean(),
    "std": df[candidates].std(),
    "min": df[candidates].min(),
    "max": df[candidates].max(),
}).sort_values("fill_rate", ascending=False)

print(f"\nTop 50 features by fill rate:")
coverage.head(50)

## 3. IC Analysis

Compute yearly IC (sector-neutral) for all candidates vs `forward_return_1y`.  
Then derive: mean IC, IC std, ICIR (= mean/std), Newey-West t-stat, % positive IC years.

In [ ]:
ic_results = []
yearly_ics = {}  # feat -> Series of yearly ICs

for feat in candidates:
    ic_series = compute_yearly_ic(df, feat, RET_COL, sector_neutral=True)
    if len(ic_series) < 3:
        continue
    mean_ic = ic_series.mean()
    std_ic = ic_series.std()
    icir = mean_ic / std_ic if std_ic > 1e-8 else 0.0
    nw_t = newey_west_tstat(ic_series)
    pct_pos = (ic_series > 0).mean()
    fill = df[feat].notna().mean()

    ic_results.append({
        "feature": feat,
        "mean_IC": mean_ic,
        "std_IC": std_ic,
        "ICIR": icir,
        "NW_t": nw_t,
        "pct_positive_IC": pct_pos,
        "fill_rate": fill,
        "abs_ICIR": abs(icir),
    })
    yearly_ics[feat] = ic_series

ic_df = pd.DataFrame(ic_results).sort_values("abs_ICIR", ascending=False).reset_index(drop=True)
print(f"Features with sufficient IC data: {len(ic_df)}")
print(f"Mean |ICIR| across all features: {ic_df['abs_ICIR'].mean():.3f}")

## 4. IC Heatmap — Top 20 Features by |ICIR|

In [ ]:
top20 = ic_df.head(20)["feature"].tolist()

heatmap_data = pd.DataFrame({f: yearly_ics[f] for f in top20})
heatmap_data = heatmap_data.sort_index()

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    heatmap_data.T,
    cmap="RdBu_r",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "IC (Spearman)"},
)
ax.set_title("Yearly IC Heatmap — Top 20 Features by |ICIR| (Sector-Neutral)", fontsize=13)
ax.set_xlabel("Fiscal Year")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

## 5. Feature Rankings — Top 30 by |ICIR|

In [ ]:
display_cols = ["feature", "mean_IC", "std_IC", "ICIR", "NW_t", "pct_positive_IC", "fill_rate"]
ic_df[display_cols].head(30)

## 6. FDR-Corrected Significance

Convert NW t-stats to p-values, then apply Benjamini-Hochberg FDR correction at α=0.05.

In [ ]:
n_years = ic_df["feature"].apply(lambda f: len(yearly_ics[f]))
pvalues = pd.Series(
    2 * (1 - stats.t.cdf(ic_df["NW_t"].abs().values, df=n_years.values - 1)),
    index=ic_df.index,
)

significant = bh_fdr_correction(pvalues, alpha=0.05)
ic_df["significant_FDR"] = significant.values

n_sig = ic_df["significant_FDR"].sum()
print(f"Features significant after BH-FDR correction (α=0.05): {n_sig} / {len(ic_df)}")
print(f"\nSignificant features (top 20 by |ICIR|):")
ic_df.loc[ic_df["significant_FDR"], display_cols].head(20)

## 7. Factor Group Breakdown

Group features by factor category (Value / Quality / Momentum / Growth / Fraud Risk / Other)  
using `alpha.horizon_router.FEATURE_FACTOR_GROUPS`.

In [ ]:
ic_df["factor_group"] = ic_df["feature"].map(FEATURE_FACTOR_GROUPS).fillna("Other")

group_stats = (
    ic_df.groupby("factor_group")
    .agg(
        n_features=("feature", "count"),
        mean_abs_ICIR=("abs_ICIR", "mean"),
        mean_IC=("mean_IC", "mean"),
        n_significant=("significant_FDR", "sum"),
        pct_significant=("significant_FDR", "mean"),
    )
    .sort_values("mean_abs_ICIR", ascending=False)
)

print("Factor group summary:")
group_stats

## Summary

This notebook demonstrates the research infrastructure working end-to-end:  
- `research.ic_engine` computes sector-neutral ICs with proper statistical testing  
- `alpha.horizon_router.FEATURE_FACTOR_GROUPS` categorizes features into the 5-factor model  
- `modeling.train.get_candidates` correctly filters the feature universe  

Architecture V2 is functional for production research.